# Material Control & Logistics – Cheatsheet Conceptual
## Proyecto: Dynamic Scheduling & Performance Control of a Real Wind Farm Construction Project (C2011-13)

**Objetivo de este documento**  
Darte el marco teórico sólido para defender en entrevista (MEINS) el bloque de control de materiales y logística.  
No es un tutorial de código. Es el **idioma de negocio** que un Project Planner & Data Analyst debe hablar cuando se habla de equipos industriales críticos.

---

### Por qué este tema es decisivo para MEINS
MEINS fabrica y entrega equipos industriales para proyectos de energías renovables.  
Su dolor principal no es solo “hacer el Gantt”, sino **evitar que un retraso de suministro de un componente crítico (nacelle, torre, transformador, pala…) genere un cuello de botella que arrastre todo el cronograma y dispare el coste**.  

Tu valor diferencial: has modelado exactamente esa lógica sobre datos reales de un parque eólico, la has conectado con EVM y has generado alarmas proactivas.

# 1. ¿Qué es realmente el Material Control en un proyecto industrial?

El **Material Control** (o Material & Logistics Control) es el proceso de planificar, seguir y asegurar que los materiales y equipos críticos llegan a obra **en la cantidad correcta, en la calidad correcta y en el momento correcto**, de forma que no detengan la ejecución.

No es solo “comprar y almacenar”. Es **gestión del riesgo de suministro** integrado con el cronograma.

| Concepto | Qué significa en la práctica | Ejemplo en parque eólico |
| :--- | :--- | :--- |
| **Long-lead item** | Componente cuyo plazo de fabricación + transporte es muy largo (meses) | Nacelle, pala, torre seccional, transformador principal |
| **Critical material** | Material cuyo retraso impacta directamente en la ruta crítica o en actividades de alto valor | Nacelle (sin ella no hay montaje de turbina) |
| **Material Readiness Date** | Fecha en la que el material debe estar disponible en obra (o en puerto/almacén intermedio) para no retrasar la actividad de montaje | Si el montaje de torre empieza el 15/06, la torre debe estar lista el 01/06 (con buffer) |
| **Lead Time** | Tiempo desde el pedido hasta la entrega en el punto de uso | Fabricación + transporte marítimo + aduanas + transporte terrestre |
| **Buffer / Safety stock temporal** | Días de colchón que se dejan entre la llegada prevista y el inicio de la actividad | 10-21 días típicos en equipos grandes |
| **Supply Risk** | Probabilidad + impacto de que el material no llegue a tiempo | Alta para componentes fabricados en Asia o con pocos proveedores |

**Regla de oro del Planner:**  
> Un retraso de suministro de un long-lead item no se recupera fácilmente. Se propaga a montaje, a comisionamiento y a la fecha de energización. Por eso se controla con la misma (o mayor) disciplina que el EVM.

# 2. Material Control vs Schedule Control vs Cost Control (EVM)

Los tres se retroalimentan. Un Planner maduro no los mira por separado.

| Dimensión | Pregunta que responde | Indicador típico | Qué pasa si falla el material |
| :--- | :--- | :--- | :--- |
| **Schedule** | ¿Vamos a tiempo? | SPI, SV, Critical Path | La actividad de montaje no puede empezar → SPI cae |
| **Cost** | ¿Vamos según presupuesto? | CPI, CV, EAC | Stand-by de grúa, personal, penalizaciones → CPI cae |
| **Material** | ¿Tenemos lo necesario para ejecutar? | Material Status, Supply Risk Score, Alarmas | Es la **causa raíz** más frecuente de los dos anteriores en proyectos de equipos industriales |

**Conexión clave que debes verbalizar en entrevista:**

> “En el proyecto C2011-13 vimos desviaciones claras de SPI y CPI. Al modelar el Material Control descubrimos que gran parte del riesgo estaba concentrado en un número reducido de long-lead items (Nacelle + Towers). Eso permite priorizar acciones correctivas donde realmente duele, en lugar de mirar 167 actividades por igual.”

# 3. Conceptos operativos que usamos en el Notebook 05

## 3.1 Clasificación de materiales críticos

No todas las actividades son iguales. Se etiquetan según tipo de material y criticidad:

| Material_Type | Criticidad típica | Lead Time orientativo | Por qué es crítico |
| :--- | :--- | :--- | :--- |
| **Nacelle** | Critical | 180-300 días | Corazón de la turbina. Un solo retraso para todo el parque |
| **Blade / Pala** | Critical | 120-200 días | Dimensiones especiales, transporte complejo |
| **Tower** | Critical | 90-150 días | Secciones pesadas, logística de transporte especial |
| **Transformer** | High / Critical | 120-180 días | Equipo eléctrico principal, pocos fabricantes |
| **Crane** | High | 30-60 días (movilización) | Recurso de montaje. Sin grúa no hay izado |
| **Cable** | Medium-High | 60-120 días | Menos “sexy” pero puede bloquear conexión |
| **Foundation** | Medium | 30-75 días | Más controlable (obra civil local) |

En el Notebook 05 aplicamos un diccionario de reglas (keywords + prioridad) sobre los nombres reales de las 167 actividades y etiquetamos **44 actividades críticas** (~26%).

## 3.2 Material Readiness Date (la fecha que realmente importa)

**Definición:**  
Fecha en la que el material debe estar disponible en el punto de uso (obra / puerto / laydown area) para no comprometer el inicio de la actividad de montaje.

**Lógica simple y defendible:**

```
Material_Readiness_Date = Baseline_Start_Actividad – Buffer_Days
```

Donde:
- `Baseline_Start_Actividad` = fecha planificada de inicio de la actividad que consume el material
- `Buffer_Days` = colchón de seguridad (en el modelo usamos valores realistas por tipo de material: 14-21 días típicamente)

**Por qué no usamos solo el Lead Time desde el pedido:**  
Porque el dataset histórico no tiene fechas reales de pedido. Usamos la lógica inversa (desde la necesidad hacia atrás), que es exactamente lo que hace un Planner cuando construye el plan de suministros a partir del Gantt.

## 3.3 Supply Risk Score y Alarmas de Material

Como no tenemos el estado real de los pedidos del proveedor (dataset histórico), construimos una **simulación realista y defendible** basada en:

1. Criticidad del material
2. Posición temporal de la actividad (cuánto de cerca está del montaje)
3. Comportamiento real del proyecto (el C2011-13 tuvo problemas claros de plazo y coste)

| Material_Status | Significado | Acción típica del Planner |
| :--- | :--- | :--- |
| **On Track** | Suministro dentro de buffer | Seguimiento rutinario |
| **At Risk** | Buffer erosionándose | Intensificar seguimiento con proveedor, preparar plan B |
| **Delayed** | Ya fuera de buffer | Escalado, posibles medidas de aceleración o re-secuenciación |
| **Critical Delay** | Impacto inminente en ruta crítica o en actividades de alto valor | Crisis meeting, posibles órdenes de cambio, replanificación |

| Material_Alarm | Criterio | Intensidad |
| :--- | :--- | :--- |
| **OK** | On Track | Verde |
| **WARNING** | At Risk o Delayed | Ámbar |
| **CRITICAL** | Critical Delay o score muy alto | Rojo – requiere acción inmediata |

**KPIs que generamos (y que debes saber explicar):**
- % de materiales críticos On Track / At Risk / Delayed / Critical Delay
- Número de alarmas CRITICAL y WARNING
- Supply Risk Score medio y máximo
- Score específico de la Nacelle (el componente más singular)

# 4. La mentalidad proactiva que busca MEINS

Un Project Planner & Data Analyst no espera a que el material no llegue para reaccionar. Su trabajo es:

1. **Identificar** los long-lead items y las actividades que dependen de ellos.
2. **Calcular** la Material Readiness Date a partir del cronograma.
3. **Monitorizar** el riesgo de suministro con indicadores tempranos.
4. **Generar alarmas** antes de que el retraso se convierta en desviación de SPI/CPI.
5. **Proponer acciones correctivas** priorizadas (no 40 acciones, sino las 5-7 que realmente mueven la aguja).
6. **Escalar** de forma clara al Project Manager / Director de Proyecto cuando el riesgo supera el umbral.

Esto es exactamente lo que modelamos en la secuencia 04 → 05 → 06.

**Frase de entrevista que debes interiorizar:**

> “Mi enfoque no es solo medir el pasado (EVM clásico). Es anticipar los cuellos de botella de suministro de equipos críticos y dar al equipo de proyecto tiempo de reacción. Eso es lo que diferencia a un Planner reactivo de uno proactivo.”

# 5. Cómo se ve en el proyecto real C2011-13

Datos que ya tienes y debes manejar con soltura:

| Métrica | Valor en el modelo |
| :--- | :--- |
| Total actividades | 167 |
| Actividades críticas de material | 44 (26,3 %) |
| Distribución principal | Tower 22 · Foundation 16 · Cable 5 · Nacelle 1 |
| % On Track | ~4,5 % |
| % At Risk | ~22,7 % |
| % Delayed | ~52,3 % |
| % Critical Delay | ~20,5 % |
| Alarmas CRITICAL | 9 |
| Alarmas WARNING | 33 |
| Supply Risk Score medio | 68,5 |
| Nacelle Risk Score | 77 |

**Interpretación de negocio (no de código):**  
El proyecto tenía un riesgo de suministro estructuralmente alto, coherente con los retrasos y sobrecostes reales que sufrió. El modelo no “inventa” problemas: refleja de forma cuantitativa lo que el histórico ya mostraba cualitativamente.

# 6. Relación con tu experiencia previa (la ventaja competitiva)

Tú no vienes de logística de turbinas. Vienes de:
- **Controller / ADE** → disciplina de seguimiento, KPIs, desviaciones, reporting.
- **Promoción-construcción inmobiliaria** → colaborabas con el jefe de obra en el **cálculo de certificaciones de obra** (eso es Earned Value aplicado en la práctica: valorar lo ejecutado para certificar).
- **Máster Data Science** → capacidad de modelar, automatizar y visualizar.

**Puente que debes construir en la entrevista:**

> “En obra yo ya participaba en la medición de lo ejecutado para las certificaciones. Eso es conceptualmente el Earned Value. Ahora he añadido la capa de Material Control porque en proyectos de equipos industriales el suministro es la variable que más frecuentemente destroza el EVM. He unido las dos perspectivas: la del controller que mide y la del planner que anticipa.”

Eso convierte la edad y la trayectoria no lineal en una **ventaja diferencial**, no en un handicap.

# 7. Preguntas típicas de entrevista y respuestas modelo

**P1: “¿Qué es para ti el control de materiales críticos?”**  
R: “Es el proceso de asegurar que los long-lead items llegan en el momento que el cronograma los necesita, con buffers realistas, y de generar alarmas tempranas cuando el riesgo de suministro amenaza la ruta crítica o el EVM. No es solo logística; es gestión de riesgo integrada con plazo y coste.”

**P2: “¿Cómo calcularías la fecha en la que un material debe estar en obra?”**  
R: “Parto de la fecha de inicio de la actividad que lo consume (Baseline Start), resto un buffer de seguridad dimensionado según el tipo de material y la criticidad, y obtengo la Material Readiness Date. Esa fecha es la que uso para seguir al proveedor.”

**P3: “En tu proyecto, ¿qué habrías hecho de forma diferente si hubieras tenido las herramientas de MEINS?”**  
R: “Hubiera priorizado el seguimiento intensivo de la Nacelle y de las Torres desde fases muy tempranas, habría generado un dashboard de Material Alarms semanal para el Project Manager, y habría propuesto planes de contingencia (proveedores alternativos, aceleración de transporte, re-secuenciación de montaje) antes de que el SPI se degradara de forma irreversible.”

**P4: “¿Cómo conectas Material Control con EVM?”**  
R: “Un retraso de suministro se traduce primero en una actividad que no puede empezar → el EV no crece → el SPI cae. Si además hay costes de stand-by (grúa, personal, penalizaciones), el CPI también se degrada. Por eso el Material Control es una de las mejores formas de proteger el EVM de forma proactiva.”

**P5: “¿Qué KPIs de materiales usarías en un dashboard ejecutivo?”**  
R: “Porcentaje de materiales críticos por estado (On Track / At Risk / Delayed / Critical Delay), número de alarmas CRITICAL, Supply Risk Score de los 5-7 items más críticos, y un semáforo de la Nacelle/Transformer (los más singulares). Todo ello enlazado con la evolución del SPI/CPI.”

# 8. Resumen de una página (para repasar 10 minutos antes de la entrevista)

**Material Control** = asegurar que los equipos críticos llegan a tiempo para no detener el montaje.

**Long-lead items** = Nacelle, Pala, Torre, Transformador → plazos de meses.

**Material Readiness Date** = Baseline Start – Buffer.

**Alarmas** = On Track / At Risk / Delayed / Critical Delay → WARNING / CRITICAL.

**Conexión con EVM** = retraso de material → actividad no arranca → SPI y CPI caen.

**Tu valor** = has modelado esta lógica sobre un proyecto real, has generado alarmas proactivas y sabes priorizar las pocas actividades que realmente mueven el riesgo del proyecto.

**Frase de cierre:**  
> “No espero a que el material falte. Construyo el sistema de alerta temprana que da al equipo tiempo de reacción. Eso es lo que un Planner proactivo aporta a MEINS.”

---

**Próximo paso después de interiorizar este cheatsheet:**  
Armazón del Notebook 06 (Acciones correctivas + Lecciones aprendidas + Escalado a nivel portfolio) y cómo se presenta este bloque en el storytelling de candidatura.

*Documento preparado como material de consulta rápida para la candidatura a Project Planner & Data Analyst en MEINS.*